# Verify a demopack

This notebook inspects a portable demopack before optionally importing it into a flow. Creating a demopack remains the responsibility of `Create_Demopack.py`.

In [ ]:
import sys
sys.path.append('..')

from pathlib import Path
from pprint import pprint
import yaml

from exp_run_config import Config
from demopack import group_chooser_sp_bc_standard, import_demopack

Config.PROJECTNAME = 'BerryPicker'

## Choose a demopack

Set the name of an existing demopack. This cell only locates the pack; it does not copy or modify data.

In [ ]:
demopack_name = 'random-both-cameras-video'
demopack_path = Path(Config()['demopacks_path'], demopack_name).expanduser()
print(demopack_path)

## Inspect the contents

A demopack contains demonstration directories and may include a run manifest named after the demopack.

In [ ]:
demo_dirs = sorted(
    path for path in demopack_path.iterdir()
    if path.is_dir() and not path.name.startswith(('.', '_'))
)
manifest = demopack_path / f'{demopack_name}.yaml'

print(f'Demonstrations: {len(demo_dirs)}')
print(f'Manifest: {manifest} ({manifest.exists()})')
print([path.name for path in demo_dirs])

## Inspect demonstration metadata

Metadata records the cameras, duration, and whether the demonstration stores image files or video.

In [ ]:
metadata = [
    yaml.safe_load((demo_dir / '_metadata.yaml').read_text())
    for demo_dir in demo_dirs
]
cameras = sorted({camera for item in metadata for camera in item['cameras']})

print('Cameras:', cameras)
pprint(metadata[0])

## Preview the data split

This shows how the standard chooser would allocate the pack's demonstrations without writing any files.

In [ ]:
copies, selection = group_chooser_sp_bc_standard([path.name for path in demo_dirs])
for group, names in selection.items():
    print(f'{group}: {len(names)} demonstrations')

## Optionally import the demopack

Importing copies demonstrations and writes configuration into the selected flow. Set `IMPORT_DEMOPACK` to `True` only when that is intended.

In [ ]:
IMPORT_DEMOPACK = False

if IMPORT_DEMOPACK:
    flow_name = 'Verify_Demopack'
    flow_path = Path(Config()['flows_path'], flow_name).expanduser()
    Config().set_exprun_path(flow_path / 'expruns')
    Config().set_results_path(flow_path / 'results')

    selection = import_demopack(demopack_path, group_chooser_sp_bc_standard)
    pprint(selection)